# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the Croissant dataset.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Identifier: {dataset.metadata.identifier}")
print(f"Date Published: {dataset.metadata.date_published}")

## 2. Data Overview
Review available record sets, fields, and their associated `@id` values.

**Tip:** See the Croissant schema documentation for more about the Croissant structure: https://mlcommons.github.io/croissant/

In [ ]:
# List all record sets with their @id and name
print("Available Record Sets:")
for rset in dataset.record_sets:
    print(f"- @id: {rset.id}, name: {getattr(rset, 'name', None)}")

# For each record set, print fields and columns (if present)
for rset in dataset.record_sets:
    print(f"\nRecord Set '@id': {rset.id}")
    if hasattr(rset, 'fields') and rset.fields:
        print("Fields:")
        for f in rset.fields:
            print(f"    - field @id: {f.id}, name: {getattr(f, 'name', None)}, data_type: {getattr(f, 'data_type', None)}")
            # If columns exist (i.e. tabular data), print them as well
            if hasattr(f, 'columns') and f.columns:
                for col in f.columns:
                    print(f"        * column @id: {col.id}, name: {getattr(col, 'name', None)}, data_type: {getattr(col, 'data_type', None)}")
    elif hasattr(rset, 'columns') and rset.columns:  # sometimes columns are directly on the record set
        print("Columns:")
        for col in rset.columns:
            print(f"    * column @id: {col.id}, name: {getattr(col, 'name', None)}, data_type: {getattr(col, 'data_type', None)}")
    else:
        print("(No fields or columns defined)")

## 3. Data Extraction
Load records from a specific record set using the record set and field `@id` values shown above.

_We'll extract all tabular record sets to pandas DataFrames for analysis (using their `@id`)._

In [ ]:
# Collect all tabular record sets (that yield data)
available_record_set_ids = [rset.id for rset in dataset.record_sets]
print("\nAvailable record set @ids:")
for rsid in available_record_set_ids:
    print(f" - {rsid}")

dataframes = {}
for record_set_id in available_record_set_ids:
    # records() is a generator of dicts; convert to DataFrame
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"\nLoaded DataFrame for record_set {record_set_id}, shape: {df.shape}")
        print(f"Columns: {list(df.columns)}")

## 4. Exploratory Data Analysis (EDA)
Let's select a record set with tabular clinical data and perform simple EDA.

Typical EDA steps:
- Filter records based on a numeric field (e.g., age, number of months between diagnoses, etc.)
- Normalize numeric fields
- Group and summarize by key attributes (e.g., sex or diagnosis type)

_Use the `@id`s of the record set and field as shown above._

In [ ]:
# We'll pick the first available tabular record set for this demo
if dataframes:
    record_set_id = next(iter(dataframes.keys()))
    df = dataframes[record_set_id]
    print(f"Using record set {record_set_id} for EDA. Columns: {df.columns.tolist()}")
    # Select a numeric field by inspecting the columns (for demo, use 'Age' or similar if present)
    numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    # Fallback: Try common column names
    for candidate in ['Age', 'age', 'Interval_months', 'interval_months', 'Interval', 'interval']:
        if candidate in df.columns:
            numeric_field = candidate
            break
    else:
        numeric_field = numeric_field_candidates[0] if numeric_field_candidates else df.columns[0]
    group_field_candidates = [col for col in df.columns if col.lower() in ['sex','gender','msi_status','site','anatomic_site'] or df[col].dtype==object]
    group_field = group_field_candidates[0] if group_field_candidates else None

    print(f"Selected numeric field: {numeric_field}")
    if group_field:
        print(f"Selected group field: {group_field}")
    else:
        print("No group field found.")

    # Filter by numeric field (> threshold)
    if pd.api.types.is_numeric_dtype(df[numeric_field]):
        threshold = df[numeric_field].quantile(0.75)
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by group_field and show mean (if possible)
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field, observed=True)[numeric_field].mean().reset_index()
            print(f"Mean {numeric_field} grouped by {group_field}:")
            display(grouped_df)
    else:
        print(f"Field '{numeric_field}' is not numeric. EDA limited.")
else:
    print("No tabular record sets with data available for EDA.")

## 5. Visualization
Visualize the selected numeric field distribution, and grouped means by category (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

if dataframes and pd.api.types.is_numeric_dtype(df[numeric_field]):
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.show()

    if group_field and group_field in df.columns:
        group_means = df.groupby(group_field, observed=True)[numeric_field].mean().reset_index()
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field, y=numeric_field, data=group_means)
        plt.title(f'Mean {numeric_field} by {group_field}')
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=30)
        plt.show()
else:
    print("Not enough numeric data for plotting.")

## 6. Conclusion

- The FAIR² dataset was loaded and explored via its Croissant schema with the `mlcroissant` Python library.
- All data elements (record sets, fields, columns) were referenced by their `@id` throughout.
- The extracted tabular data supports common EDA tasks such as filtering, normalization, and grouping by clinical attributes.
- Visualizations provide further insight into the distributions of clinical or molecular attributes in the cancer survivor cohort.

**Next Steps:**
- Use other fields or record sets for further analysis as required
- Integrate the FAIR² dataset into clinical and molecular research pipelines
- Consult the schema's `@id` definitions for precise programmatic referencing